# RAG Principle Model — Notebook


**Prompt modes:**
- `rag_zeroshot` — principles + k similar texts
- `rag_oneshot` — principles + one labelled example + k similar texts
- `zeroshot` — principles only (no RAG)
- `oneshot` — principles + one labelled example (no RAG)

In [6]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "principle_model").exists():
    project_root = (project_root / ".." / "..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from principle_model.predict import predict
from principle_model.evaluate import evaluate

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


In [7]:
test_csv       = str(project_root / "data/split/essays/test.csv")
principles_dir = str(project_root / "data/principles")
res_dir        = str(project_root / "result")
log_dir        = str(project_root / "log")

# Fine-tuned artifacts are resolved automatically from
# principle_model/predict.py (FINETUNED_ARTIFACTS_DIR / FINETUNED_MODEL_DIR).
# Uncomment below to override the default path:
# vector_db_dir = str(project_root / "models/rag_artifacts")

model_name     = "gpt-4o-mini"
max_new_tokens = 512
temperature    = 0.0

# prompt_mode: "rag_zeroshot" | "rag_oneshot" | "zeroshot" | "oneshot"
prompt_mode    = "rag_oneshot"
top_k          = 3

test_df = pd.read_csv(test_csv)

print(f"Test  : {len(test_df):>4} rows")
print(f"Mode  : {prompt_mode}  |  top_k={top_k}")

Test  :  247 rows
Mode  : rag_oneshot  |  top_k=3


## Run prediction

In [8]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    principles_dir = principles_dir,
    top_k          = top_k,
)

print(f"\nDone in {run_time:.1f}s")
print(f"Predictions saved to: {prediction_csv}")

[retriever] Using fine-tuned artifacts: F:\std\GR\code\model_x_ocean\models\rag_artifacts
[predict] RAG retriever loaded (top_k=3).
[predict] 247 records | mode=rag_oneshot | model=gpt-4o-mini
[retriever] Loaded fine-tuned FAISS index (1974 vectors).
  [predict] 10/247 done.
  [predict] 20/247 done.
  [predict] 30/247 done.
  [predict] 40/247 done.
  [predict] 50/247 done.
  [predict] 60/247 done.
  [predict] 70/247 done.
  [predict] 80/247 done.
  [predict] 90/247 done.
  [predict] 100/247 done.
  [predict] 110/247 done.
  [predict] 120/247 done.
  [predict] 130/247 done.
  [predict] 140/247 done.
  [predict] 150/247 done.
  [predict] 160/247 done.
  [predict] 170/247 done.
  [predict] 180/247 done.
  [predict] 190/247 done.
  [predict] 200/247 done.
  [predict] 210/247 done.
  [predict] 220/247 done.
  [predict] 230/247 done.
  [predict] 240/247 done.
[predict] Finished in 2767.04s -> F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\predictions.csv

Done in

## Evaluate

In [9]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")

Loaded predictions from F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\predictions.csv
Saved evaluation summary to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\evaluation_summary.csv
Saved Openness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\Openness_classification_report.txt
Saved Conscientiousness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\Conscientiousness_classification_report.txt
Saved Extraversion report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\Extraversion_classification_report.txt
Saved Agreeableness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\Agreeableness_classification_report.txt
Saved Neuroticism report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\rag_oneshot\20260503-182339\Neuroticism_classification_report.txt
Summary CSV: F:\std\GR\code\model_x

## Results summary

In [10]:
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

,trait,n_samples,accuracy,macro_f1,weighted_f1
0,Extraversion,247,0.591093,0.587196,0.586059
1,Neuroticism,247,0.582996,0.536805,0.537397
2,Openness,247,0.562753,0.551573,0.549567
3,Agreeableness,247,0.534413,0.529196,0.525384
4,Conscientiousness,247,0.522267,0.390047,0.386598
